# UM4MA266, Numerical optimization and Data science
## STEREO Project

Student : SIDATE Mathieu<br>
Sorbonne Université, 2025-2026

## Research phase

### 1. Implementation de la méthode de block matching 

#### 1.1. Calcul du SSD (Sum of Squared differences)

On a que,
$$
C(x,y,d) = \displaystyle\sum_{(i,j)\in\cal{W}} |{I}_{L}(x+i, y+j) - {I}_{R}(x+i+d, y+j)|^2, \tag{1}
$$
avec $\cal{W}$ notre window (ou bloc) de pixels,<br><br> 
${I}_{L}$ et ${I}_{R}$ nos images gauche droite qui renvoient ici une valeur spécifique à un pixel (généralement la luminosité sur une image mis en niveau de gris).

In [2]:
import numpy as np

<span style="color:green"> <u>Thoughts:</u> 
- Comment on gère la window; centrée en un point, commence au coin supérieur gauche ?<br>
- Doit-on, pour chaque pixel dans une fenêtre, calculer le SSD local ou, pour chaque pixel, créer une fenêtre de taille window et calculer le SSD ? 
</span>

On essaie avec centré en un point et pour chaque pixel, window ssd. (ça à l'air d'être comme ça avec OpenCV)

<span style="color:green"> <u>2nd Thought:</u>
Dans la question 1, de ce que j'ai compris, on regarde TOUS les pixels de l'image et créer une fenêtre à chaque fois
</span>

In [31]:
# Méthode de SSD local avec fenêtre

def ssd(l_img, r_img, d, window):
    """
    Args :
        [l_img] matrice de pixels (dim = résolution image) de l'image gauche ayant pour valeur la luminosité
        [r_img] // pour l'image droite
        [d] disparité (ie. écart horizontal entre 2 pixels)
        [window] taille de notre fenêtre carrée de pixel

    Outputs :
        [cost] coût de correspondance (+ il est faible + les images à translation d près sont similaires)

    Assumptions :
        - dim(l_img) = dim(r_img)
        - taille de window < taille d'image
        - disparité d < largeur image
    """

    h, w = l_img.shape  # hauteur (height) , largeur (width) des images
    half_win = window // 2
    l_img , r_img = np.array(l_img) , np.array(r_img)
    cost = 0
    
    # On parcourt tous les pixels de l'image qui peuvent acceuillir une window centré dans l'image
    for y in range(half_win, h-half_win):
        for x in range(half_win, w-half_win):
            
            I_ls = np.array(l_img[y-half_win:y+half_win, x-half_win:x+half_win])             # fenêtre centrée en (x,y)   
            
            if x+d-half_win >= 0 and x+d+half_win <= w:
                I_rs = np.array(r_img[y-half_win:y+half_win, x+d-half_win:x+d+half_win])       # fenêtre centrée en (x+d, y) à condition la fenêtre existe
            else:
                I_rs = np.zeros(I_ls.shape[0])
                
            cost += np.sum((I_ls - I_rs)**2)
    
    return cost 

nb : faire un petit schéma pour expliquer choix de fenêtre

In [38]:
A = np.random.rand(720, 1280)
B = np.random.rand(720, 1280)
ssd(A, B, 70, 30)

np.float64(136571743.6908929)

ça a l'air de marcher en ayant pris l'exemple d'images en HD aléatoire

### 2. Régularisation du problème

Notre but est de trouver $d^*$ tel que $d^* = \displaystyle \arg\min_{d \in \mathbb{N}^*} C_{tot}(d)$

<span style="margin-left:2em;"> où, $C_{tot}(d) = \displaystyle \sum_{(x,y) \in I} C(x,y,d)$</span>
<span style="margin-left:2em;"> avec $I$ la dimension de la matrice de pixels. </span>

On aimerait utiliser les techniques d'optimisation usuelles sur $C_{tot}$ pour trouver un minimiseur mais un problème premier étant que la fonction n'est pas régulière.

En effet, d est à valeur dans $\mathbb{N}$ et on ne sait rien de la différentiabilité de la fonction $I_{R}$ qui est cruciale dans le calcul de différentielle de $(1)$.